In [0]:
%sql
select * from dev.`1_bronze`.salesorderdetail

In [0]:
start_row = 81    
end_row = 500    
batch_size = 5       

adls_base_path = "abfss://kafkainput@indiaprojectstorage.dfs.core.windows.net/topics/salesorderdetail".rstrip("/")
# Staging folder (removed at the end); the connector should only pick up salesorderdetail<N>.json files
tmp_path = f"{adls_base_path}/_staging"


from pyspark.sql import Window
from pyspark.sql.functions import row_number, ceil, col, lit, struct, collect_list, sort_array, transform, to_json

df = spark.table("`sales-northindia-db`.saleslt.salesorderdetail")

# Deterministic ordering so each batch file always contains the same rows
window_spec = Window.orderBy("SalesOrderID", "SalesOrderDetailID")
df_with_row = df.withColumn("_row_num", row_number().over(window_spec))

df_filtered = df_with_row.filter(
    (col("_row_num") >= start_row) & (col("_row_num") <= end_row)
)

print(f"Total rows in table : {df.count()}")
print(f"Rows selected ({start_row} to {end_row}): {df_filtered.count()}")

df_batched = df_filtered.withColumn(
    "_batch_id",
    ceil((col("_row_num") - lit(start_row) + lit(1)) / lit(batch_size)).cast("int")
)

total_batches = df_batched.select("_batch_id").distinct().count()
print(f"Number of batches (batch_size={batch_size}): {total_batches}")

# Build one JSON array string per batch: [{...},{...}], keeping rows in _row_num order
data_cols = [c for c in df.columns]
df_arrays = df_batched \
    .groupBy("_batch_id") \
    .agg(sort_array(collect_list(struct(col("_row_num"), struct(*data_cols).alias("rec")))).alias("_rows")) \
    .select(
        "_batch_id",
        to_json(transform(col("_rows"), lambda x: x["rec"]), {"ignoreNullFields": "false"}).alias("value")
    )

# Write all batches once to a staging folder (one partition folder per batch, one file each)
df_arrays \
    .repartition("_batch_id") \
    .write \
    .mode("overwrite") \
    .partitionBy("_batch_id") \
    .text(tmp_path)

# Move each batch's part file to salesorderdetail<N>.json, then remove the staging folder
written_files = []
for folder in dbutils.fs.ls(tmp_path):
    if not folder.name.startswith("_batch_id="):
        continue
    batch_id = int(folder.name.rstrip("/").split("=")[1])
    part_files = [f.path for f in dbutils.fs.ls(folder.path) if f.name.startswith("part-") and f.name.endswith(".txt")]
    if len(part_files) != 1:
        raise RuntimeError(f"Expected 1 part file for batch {batch_id}, found {len(part_files)}")
    target_file = f"{adls_base_path}/salesorderdetail{batch_id}.json"
    dbutils.fs.rm(target_file)  # replace file from a previous run, if any
    dbutils.fs.mv(part_files[0], target_file)
    written_files.append((batch_id, target_file))

dbutils.fs.rm(tmp_path, True)

print(f"✅ {len(written_files)} JSON batch file(s) written to: {adls_base_path}")



In [0]:
start_row =1    
end_row = 50        
batch_size = 2      

adls_base_path = "abfss://kafkainput@indiaprojectstorage.dfs.core.windows.net/topics/salesorderheader".rstrip("/")
# Staging folder (removed at the end); the connector should only pick up salesorderdetail<N>.json files
tmp_path = f"{adls_base_path}/_staging"


from pyspark.sql import Window
from pyspark.sql.functions import row_number, ceil, col, lit, struct, collect_list, sort_array, transform, to_json

df = spark.table("`sales-northindia-db`.saleslt.salesorderheader")

# Deterministic ordering so each batch file always contains the same rows
window_spec = Window.orderBy("SalesOrderID")
df_with_row = df.withColumn("_row_num", row_number().over(window_spec))

df_filtered = df_with_row.filter(
    (col("_row_num") >= start_row) & (col("_row_num") <= end_row)
)

print(f"Total rows in table : {df.count()}")
print(f"Rows selected ({start_row} to {end_row}): {df_filtered.count()}")

df_batched = df_filtered.withColumn(
    "_batch_id",
    ceil((col("_row_num") - lit(start_row) + lit(1)) / lit(batch_size)).cast("int")
)

total_batches = df_batched.select("_batch_id").distinct().count()
print(f"Number of batches (batch_size={batch_size}): {total_batches}")

# Build one JSON array string per batch: [{...},{...}], keeping rows in _row_num order
data_cols = [c for c in df.columns]
df_arrays = df_batched \
    .groupBy("_batch_id") \
    .agg(sort_array(collect_list(struct(col("_row_num"), struct(*data_cols).alias("rec")))).alias("_rows")) \
    .select(
        "_batch_id",
        to_json(transform(col("_rows"), lambda x: x["rec"]), {"ignoreNullFields": "false"}).alias("value")
    )

# Write all batches once to a staging folder (one partition folder per batch, one file each)
df_arrays \
    .repartition("_batch_id") \
    .write \
    .mode("overwrite") \
    .partitionBy("_batch_id") \
    .text(tmp_path)

# Move each batch's part file to salesorderdetail<N>.json, then remove the staging folder
written_files = []
for folder in dbutils.fs.ls(tmp_path):
    if not folder.name.startswith("_batch_id="):
        continue
    batch_id = int(folder.name.rstrip("/").split("=")[1])
    part_files = [f.path for f in dbutils.fs.ls(folder.path) if f.name.startswith("part-") and f.name.endswith(".txt")]
    if len(part_files) != 1:
        raise RuntimeError(f"Expected 1 part file for batch {batch_id}, found {len(part_files)}")
    target_file = f"{adls_base_path}/salesorderheader{batch_id}.json"
    dbutils.fs.rm(target_file)  # replace file from a previous run, if any
    dbutils.fs.mv(part_files[0], target_file)
    written_files.append((batch_id, target_file))

dbutils.fs.rm(tmp_path, True)

print(f"✅ {len(written_files)} JSON batch file(s) written to: {adls_base_path}")

